In [1]:
# ╔══════════════════════════════════════════════════════════╗
# ║   ⚙  J.A.R.V.I.S  —  BOOT SEQUENCE                     ║
# ║   STARK INDUSTRIES | SUPERTONIC-3 TTS ARMORY            ║
# ╚══════════════════════════════════════════════════════════╝
# Run every cell top-to-bottom. Each cell is self-contained
# and will display a status panel on completion.

from IPython.display import display, HTML, Audio, clear_output
import time

# ─── Global Iron-Man HTML builder ─────────────────────────
def stark_panel(body, title="J.A.R.V.I.S", accent="#CC0000"):
    return f"""
<div style="
    background:linear-gradient(160deg,#080808 0%,#150303 60%,#080808 100%);
    border:2px solid {accent};
    border-radius:6px; padding:18px 24px; margin:8px 0;
    font-family:'Courier New',monospace; color:#FFD700;
    box-shadow:0 0 22px {accent}66, 0 0 60px #00000099;
    max-width:860px;">
  <div style="font-size:.62em;letter-spacing:5px;color:{accent};
       border-bottom:1px solid {accent};padding-bottom:7px;margin-bottom:12px;
       font-weight:bold;">
    ⚙ STARK INDUSTRIES ─── {title} ───
  </div>
  {body}
  <div style="text-align:center;margin-top:12px;color:{accent};
       font-size:.6em;letter-spacing:3px;border-top:1px solid {accent};
       padding-top:7px;">
    ── AUTHORIZED PERSONNEL ONLY ── STARK INDUSTRIES R&amp;D ──
  </div>
</div>"""

def log(label, msg, color="#FFD700"):
    return (f'<div style="font-size:.83em;line-height:2;">'
            f'<span style="color:#00BFFF;">[ {label:<10} ]</span> '
            f'<span style="color:{color};">&#9658; {msg}</span></div>')

boot_body = """
<div style="text-align:center;margin-bottom:14px;">
  <span style="font-size:2.6em;font-weight:900;color:#CC0000;
        letter-spacing:6px;text-shadow:0 0 18px #CC0000;">
    &#9889; J.A.R.V.I.S
  </span><br>
  <span style="font-size:.74em;color:#FFD700;letter-spacing:4px;">
    JUST A RATHER VERY INTELLIGENT SPEECH-SYNTHESIS SYSTEM
  </span>
</div>
""" + "".join([
    log("REACTOR",  "Arc Reactor Power ─ NOMINAL (100%)",      "#00FF7F"),
    log("ENGINE",   "Supertonic-3 TTS Core ─ ARMING...",       "#FFD700"),
    log("LANGS",    "31 Languages ─ हिन्दी + English DUAL-ACTIVE","#FFD700"),
    log("VOICES",   "Profiles M1–M5 | F1–F5 ─ ON STANDBY",    "#FFD700"),
    log("TAGS",     "&lt;laugh&gt; &lt;breath&gt; &lt;sigh&gt; ─ EXPRESSION TAGS ARMED", "#FFD700"),
    log("CLONING",  "Voice Builder Interface ─ READY",          "#FFD700"),
    log("STATUS",   "ALL SYSTEMS NOMINAL ─ AWAITING COMMANDS",  "#00FF7F"),
])

display(HTML(stark_panel(boot_body, "BOOT SEQUENCE INITIATED")))
print("\n🔴 JARVIS online. Run all cells sequentially to arm the TTS cannon.\n")



🔴 JARVIS online. Run all cells sequentially to arm the TTS cannon.



In [2]:
# ╔══════════════════════════════════════════════════════════╗
# ║   CELL 2 ─ INSTALL ARSENAL                              ║
# ║   Installing all required packages into Colab runtime   ║
# ╚══════════════════════════════════════════════════════════╝
# Expected duration: ~60-90 seconds on a cold Colab runtime.
# All packages install from PyPI — no custom repos required.
#
# Package manifest:
#   supertonic  — Supertonic-3 TTS engine (ONNX-based, ~400MB model)
#   librosa     — Audio DSP: pitch shift, time stretch, resampling
#   soundfile   — Read/write WAV/FLAC/OGG with libsndfile bindings
#   scipy       — Scientific stack (array ops, signal processing)
#   ipywidgets  — Interactive Jupyter widgets (sliders, dropdowns)

from IPython.display import display, HTML
display(HTML(stark_panel(
    log("INSTALL", "Initialising package installer — hold on...", "#FFD700"),
    "ARSENAL LOADER"
)))

import subprocess, sys

pkgs = [
    ("supertonic",  "Supertonic-3 TTS engine + ONNX runtime"),
    ("librosa",     "Audio DSP library (pitch/speed post-processing)"),
    ("soundfile",   "WAV read/write via libsndfile"),
    ("scipy",       "Scientific computing stack"),
    ("ipywidgets",  "Interactive Colab widgets"),
]

install_log = ""
for pkg, desc in pkgs:
    print(f"  ⚙  Installing {pkg} ({desc})...")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", pkg, "-q"],
        capture_output=True, text=True
    )
    status = "✅ OK" if result.returncode == 0 else f"❌ FAILED: {result.stderr[:80]}"
    install_log += log(pkg.upper()[:10], f"{desc} ─ {status}",
                       "#00FF7F" if result.returncode == 0 else "#FF4444")
    print(f"     {status}")

display(HTML(stark_panel(
    install_log + log("STATUS", "All packages installed — ARSENAL READY", "#00FF7F"),
    "ARSENAL LOADER ─ COMPLETE", "#00FF7F"
)))


  ⚙  Installing supertonic (Supertonic-3 TTS engine + ONNX runtime)...
     ✅ OK
  ⚙  Installing librosa (Audio DSP library (pitch/speed post-processing))...
     ✅ OK
  ⚙  Installing soundfile (WAV read/write via libsndfile)...
     ✅ OK
  ⚙  Installing scipy (Scientific computing stack)...
     ✅ OK
  ⚙  Installing ipywidgets (Interactive Colab widgets)...
     ✅ OK


In [3]:
# ╔══════════════════════════════════════════════════════════╗
# ║   CELL 3 ─ IMPORT MODULES + LOAD SUPERTONIC-3 MODEL    ║
# ║   First run downloads ~400 MB from HuggingFace.         ║
# ║   Subsequent runs load from ~/.cache/supertonic3/       ║
# ╚══════════════════════════════════════════════════════════╝
# Model architecture (Supertonic-3):
#   ├── text_encoder.onnx     —  36 MB  — Text → latent embeddings
#   ├── vector_estimator.onnx — 257 MB  — Flow-matching latent estimator
#   ├── duration_predictor.onnx — 3.7 MB — Utterance duration control
#   └── vocoder.onnx          — 101 MB  — Latent → waveform (24 kHz)
# Total: ~99M parameters across all ONNX assets (~400 MB on disk)

import time, os, json, tempfile, shutil
import numpy as np
import soundfile as sf
import librosa
import scipy
import ipywidgets as widgets
from IPython.display import display, HTML, Audio, clear_output

display(HTML(stark_panel(
    log("MODULES", "Core Python modules imported", "#00FF7F") +
    log("LOADING", "Importing Supertonic SDK...", "#FFD700"),
    "MODULE IMPORT"
)))

from supertonic import TTS

display(HTML(stark_panel(
    log("SDK",    "Supertonic Python SDK — IMPORTED", "#00FF7F") +
    log("INIT",   "Initialising TTS engine (auto_download=True)...", "#FFD700") +
    log("CACHE",  "Model cache: ~/.cache/supertonic3/", "#FFD700") +
    log("NOTE",   "First run downloads ~400MB — subsequent runs are instant", "#00BFFF"),
    "MODEL LOADING"
)))

t0 = time.time()
print("⚡ Loading Supertonic-3 model (downloading if first run)...")
tts = TTS(auto_download=True)
elapsed = time.time() - t0

# Probe sample rate by synthesising a 1-char test
_probe_style = tts.get_voice_style(voice_name="M1")
_probe_wav, _ = tts.synthesize(".", voice_style=_probe_style, lang="en")
_tmp = "/content/_probe.wav"
tts.save_audio(_probe_wav, _tmp)
SAMPLE_RATE = sf.info(_tmp).samplerate
os.remove(_tmp)

display(HTML(stark_panel(
    log("ENGINE",  "Supertonic-3 TTS Engine ─ ONLINE", "#00FF7F") +
    log("RUNTIME", f"Model loaded in {elapsed:.1f}s", "#00FF7F") +
    log("SR",      f"Native sample rate: {SAMPLE_RATE} Hz", "#00FF7F") +
    log("VOICES",  "Built-in voices: M1 M2 M3 M4 M5 | F1 F2 F3 F4 F5", "#FFD700") +
    log("LANGS",   "31 ISO codes + 'na' fallback for mixed/Hinglish text", "#FFD700") +
    log("TAGS",    "<laugh>  <breath>  <sigh>  — embed anywhere in text", "#FFD700") +
    log("STATUS",  "REPULSORS CHARGED — READY TO FIRE", "#00FF7F"),
    "MODEL ONLINE", "#00FF7F"
)))
print(f"\n✅ Supertonic-3 ready at {SAMPLE_RATE} Hz. Proceed to Cell 4.\n")


⚡ Loading Supertonic-3 model (downloading if first run)...



✅ Supertonic-3 ready at 44100 Hz. Proceed to Cell 4.



In [4]:
# ╔══════════════════════════════════════════════════════════╗
# ║   CELL 4 ─ UPLOAD YOUR TEXT FILE                        ║
# ║   Upload a UTF-8 .txt file containing your Hinglish     ║
# ║   text (Devanagari + Latin mixed, with optional tags).  ║
# ╚══════════════════════════════════════════════════════════╝
# Supported expression tags (embed anywhere in your text):
#   <laugh>  — natural laughter
#   <breath> — audible breath / pause
#   <sigh>   — exasperated or content sigh
#
# Language tip for Hinglish:
#   Use lang="na"  → safest fallback for mixed Devanagari+Latin
#   Use lang="hi"  → Hindi-dominant text (pure Devanagari)
#   Use lang="en"  → English-dominant text (Latin script)
#
# Your file is read as UTF-8 — Devanagari glyphs are fully supported.

from google.colab import files as colab_files

FALLBACK_TEXT = (
    "Yaar, sun toh sahi! <breath> Aaj office mein kuch aisa hua "
    "ki mujhe khud believe nahi ho raha. मेरा boss अचानक आया और bola, "
    "'Tum log bahut achha kaam kar rahe ho!' <laugh> Main toh sochne laga, "
    "yeh koi camera prank hai kya? <sigh> Phir usne bataya ki humara project "
    "deadline se 2 din pehle complete ho gaya, aur management bahut khush hai. "
    "Seriously bhai, kabhi kabhi zindagi mein aisi surprises aati hain jo "
    "दिल खुश कर देती हैं। <breath> Toh kal party hai, sab log aana!"
)

INPUT_TEXT = FALLBACK_TEXT  # will be overwritten if file uploaded

display(HTML(stark_panel(
    log("UPLOAD", "Click the button below to upload your .txt file", "#FFD700") +
    log("FORMAT", "UTF-8 encoding required — Devanagari fully supported", "#00BFFF") +
    log("FALLBACK","Built-in Hinglish test text loaded as fallback", "#00BFFF"),
    "INTELLIGENCE UPLOAD"
)))

print("📤 Upload your .txt file (or skip — fallback test text is ready):")
try:
    uploaded = colab_files.upload()
    if uploaded:
        fname = list(uploaded.keys())[0]
        raw = uploaded[fname]
        INPUT_TEXT = raw.decode("utf-8")
        char_count = len(INPUT_TEXT)
        preview = INPUT_TEXT[:300].replace("<", "&lt;").replace(">", "&gt;")
        display(HTML(stark_panel(
            log("FILE",    f"Loaded: <b>{fname}</b>", "#00FF7F") +
            log("CHARS",   f"Total characters: {char_count}", "#00FF7F") +
            log("PREVIEW", f"<br><span style='font-size:.85em;color:#ccc;line-height:1.6'>{preview}{'...' if char_count>300 else ''}</span>", "#FFD700"),
            "FILE LOADED", "#00FF7F"
        )))
    else:
        raise ValueError("No file selected")
except Exception as e:
    display(HTML(stark_panel(
        log("FALLBACK", "No file uploaded — using built-in Hinglish test text", "#FFD700") +
        log("TEXT",     f"<br><span style='font-size:.82em;color:#ccc;line-height:1.6'>{FALLBACK_TEXT[:300].replace('<','&lt;').replace('>','&gt;')}...</span>", "#FFD700"),
        "USING FALLBACK TEXT", "#FFD700"
    )))

print(f"\n✅ Input text ready ({len(INPUT_TEXT)} chars). Proceed to Cell 5.\n")


📤 Upload your .txt file (or skip — fallback test text is ready):


Saving test_supertonic_tts.txt to test_supertonic_tts (1).txt



✅ Input text ready (527 chars). Proceed to Cell 5.



In [6]:
# ╔══════════════════════════════════════════════════════════╗
# ║   CELL 5 ─ MISSION PARAMETERS (TTS CONFIGURATION)       ║
# ║   Tune every dial before firing the repulsors.          ║
# ╚══════════════════════════════════════════════════════════╝
# PARAMETER REFERENCE:
#
#  voice_name    : M1–M5 = Male voices | F1–F5 = Female voices
#                  (ignored if a Voice Builder JSON is loaded in Cell 6)
#
#  language      : ISO 639-1 code for the dominant language.
#                  "na" = safest for mixed Hinglish (Devanagari+Latin).
#                  "hi" = Hindi dominant | "en" = English dominant.
#
#  total_steps   : Flow-matching inference steps.
#                  5  = fastest, lower quality
#                  8  = default balanced (recommended)
#                  12 = highest quality, slower
#
#  speed         : Playback speed multiplier (applied at synthesis time).
#                  0.7 = slow | 1.0 = normal | 2.0 = 2x fast
#
#  pitch_shift   : Semitone shift applied after synthesis via librosa.
#                  -6 = much lower | 0 = unchanged | +6 = much higher
#                  Note: large values may introduce artifacts.
#
#  silence_dur   : Silence (seconds) inserted between text chunks.
#                  Increase for more breathing room in long texts.
#
#  chunk_len     : Max characters per synthesis chunk.
#                  300 for most languages; 120 recommended for Korean.

import ipywidgets as widgets
from IPython.display import display, HTML

# ─── Widget Definitions ────────────────────────────────────
w_voice = widgets.Dropdown(
    options=["M1","M2","M3","M4","M5","F1","F2","F3","F4","F5"],
    value="M1",
    description="Voice:",
    style={"description_width":"120px"},
    layout=widgets.Layout(width="340px"),
)

w_lang = widgets.Dropdown(
    options=[
        ("hi — Hindi (Devanagari-heavy)",      "hi"),
        ("en — English (Latin-heavy)",          "en"),
        ("ko — Korean", "ko"), ("ja — Japanese","ja"),
        ("de — German","de"),  ("fr — French",  "fr"),
        ("es — Spanish","es"), ("ru — Russian", "ru"),
        ("ar — Arabic", "ar"),
    ],
    value="hi",
    description="Language:",
    style={"description_width":"120px"},
    layout=widgets.Layout(width="420px"),
)

w_steps = widgets.IntSlider(
    value=8, min=5, max=12, step=1,
    description="Quality (steps):",
    style={"description_width":"150px"},
    layout=widgets.Layout(width="480px"),
    continuous_update=False,
)

w_speed = widgets.FloatSlider(
    value=1.0, min=0.7, max=2.0, step=0.05,
    description="Speed:",
    readout_format=".2f",
    style={"description_width":"120px"},
    layout=widgets.Layout(width="480px"),
    continuous_update=False,
)

w_pitch = widgets.IntSlider(
    value=0, min=-8, max=8, step=1,
    description="Pitch (semitones):",
    style={"description_width":"150px"},
    layout=widgets.Layout(width="480px"),
    continuous_update=False,
)

w_silence = widgets.FloatSlider(
    value=0.3, min=0.0, max=1.5, step=0.05,
    description="Silence (s):",
    readout_format=".2f",
    style={"description_width":"120px"},
    layout=widgets.Layout(width="480px"),
    continuous_update=False,
)

w_chunk = widgets.IntSlider(
    value=300, min=50, max=500, step=10,
    description="Chunk length:",
    style={"description_width":"150px"},
    layout=widgets.Layout(width="480px"),
    continuous_update=False,
)

w_verbose = widgets.Checkbox(
    value=True,
    description="Verbose synthesis output (recommended — shows JARVIS logs)",
    layout=widgets.Layout(width="480px"),
)

# ─── Display Panel ────────────────────────────────────────
display(HTML(stark_panel(
    log("PANEL",  "Adjust all parameters below, then run Cell 6 (voice) & Cell 7 (generate)", "#FFD700") +
    log("VOICE",  "Dropdown: 5 Male (M1-M5) + 5 Female (F1-F5) built-in profiles", "#00BFFF") +
    log("LANG",   "'na' is the recommended setting for Hinglish mixed text", "#00BFFF") +
    log("QUALITY","Higher steps = better audio quality, slower synthesis", "#00BFFF") +
    log("SPEED",  "Speed is applied natively at synthesis time by Supertonic", "#00BFFF") +
    log("PITCH",  "Pitch shift is applied post-synthesis via librosa (semitones)", "#00BFFF"),
    "MISSION PARAMETERS"
)))

display(widgets.VBox([
    widgets.HTML('<b style="color:#CC0000;font-family:monospace;font-size:1em;">⚙ VOICE &amp; LANGUAGE</b>'),
    w_voice, w_lang,
    widgets.HTML('<b style="color:#CC0000;font-family:monospace;font-size:1em;">⚙ QUALITY &amp; SPEED</b>'),
    w_steps, w_speed,
    widgets.HTML('<b style="color:#CC0000;font-family:monospace;font-size:1em;">⚙ POST-PROCESSING &amp; CHUNKING</b>'),
    w_pitch, w_silence, w_chunk, w_verbose,
]))

print("\n✅ Parameters panel live. Adjust → then run Cell 6 (Voice/Cloning) → Cell 7 (Synthesize).\n")



✅ Parameters panel live. Adjust → then run Cell 6 (Voice/Cloning) → Cell 7 (Synthesize).



In [7]:
# ╔══════════════════════════════════════════════════════════╗
# ║   CELL 6 ─ VOICE CLONING STATION (OPTIONAL)             ║
# ╚══════════════════════════════════════════════════════════╝
# HOW SUPERTONIC VOICE CLONING WORKS:
# ─────────────────────────────────────────────────────────
# Supertonic-3 uses SPEAKER VECTOR EMBEDDINGS stored as JSON
# files. Voice cloning requires a Voice Builder export JSON,
# NOT a raw audio file (the ONNX package has no encoder
# for raw audio → embedding conversion).
#
# OPTION A ── Built-in voice (default, no upload needed):
#   Just leave this cell as-is. The voice selected in Cell 5
#   (w_voice dropdown) will be used automatically.
#
# OPTION B ── Voice Builder JSON (custom voice clone):
#   1. Visit: https://supertonic.supertone.ai/voice_builder
#   2. Upload a short reference clip (10–60 seconds works best)
#   3. Download the exported JSON voice style file
#   4. Run this cell and upload that JSON when prompted.
#
# The JSON is a compact vector file (~20 KB) containing the
# speaker embedding that Supertonic uses at synthesis time.

from google.colab import files as colab_files

CUSTOM_VOICE_PATH = None   # will be set if user uploads a JSON

display(HTML(stark_panel(
    log("MODE A",  "Built-in voice from Cell 5 dropdown ─ always available", "#00FF7F") +
    log("MODE B",  "Voice Builder JSON upload ─ custom cloned voice", "#FFD700") +
    log("STEP 1",  "Visit: supertonic.supertone.ai/voice_builder", "#00BFFF") +
    log("STEP 2",  "Upload 10-60s reference audio clip to Voice Builder", "#00BFFF") +
    log("STEP 3",  "Download the exported JSON voice style file", "#00BFFF") +
    log("STEP 4",  "Upload that JSON below (or skip for built-in voice)", "#00BFFF"),
    "VOICE CLONING STATION"
)))

print("📤 Upload your Voice Builder JSON file (skip/cancel for built-in voice):")
try:
    uploaded_json = colab_files.upload()
    if uploaded_json:
        jfname = list(uploaded_json.keys())[0]
        if not jfname.endswith(".json"):
            raise ValueError(f"Expected a .json file, got: {jfname}")
        json_path = f"/content/{jfname}"
        with open(json_path, "wb") as jf:
            jf.write(uploaded_json[jfname])
        # Validate it can be loaded
        test_style = tts.get_voice_style_from_path(json_path)
        CUSTOM_VOICE_PATH = json_path
        display(HTML(stark_panel(
            log("LOADED",  f"Voice JSON: <b>{jfname}</b>", "#00FF7F") +
            log("STATUS",  "Custom voice style validated and armed", "#00FF7F") +
            log("OVERRIDE","This voice will OVERRIDE the Cell 5 dropdown selection", "#FFD700"),
            "CUSTOM VOICE ARMED", "#00FF7F"
        )))
    else:
        raise ValueError("No file selected")
except Exception as e:
    display(HTML(stark_panel(
        log("SKIP",   "No Voice Builder JSON uploaded", "#FFD700") +
        log("USING",  f"Built-in voice from Cell 5 dropdown: <b>{w_voice.value}</b>", "#00FF7F"),
        "USING BUILT-IN VOICE", "#FFD700"
    )))

print(f"\n✅ Voice selection finalised. Proceed to Cell 7 (Generate TTS).\n")


📤 Upload your Voice Builder JSON file (skip/cancel for built-in voice):



✅ Voice selection finalised. Proceed to Cell 7 (Generate TTS).



In [8]:
# ╔══════════════════════════════════════════════════════════╗
# ║   CELL 7 ─ FIRE THE REPULSORS — GENERATE TTS            ║
# ║   This is the main synthesis cell. Read every log line. ║
# ╚══════════════════════════════════════════════════════════╝
# SYNTHESIS PIPELINE:
#   1. Read parameters from Cell 5 widgets
#   2. Load voice style (custom JSON or built-in)
#   3. Call tts.synthesize() — Supertonic-3 ONNX inference
#   4. Apply pitch shift via librosa.effects.pitch_shift()
#   5. Save final WAV to /content/JARVIS_TTS_output.wav
#   6. Display waveform stats + inline audio player
#
# ABOUT total_steps:
#   Supertonic uses flow-matching (like diffusion but faster).
#   More steps = more refined latent trajectory = better audio.
#   5 steps ≈ 100ms for short text; 12 steps ≈ 250ms.
#
# ABOUT speed parameter:
#   Speed is applied natively during synthesis by Supertonic's
#   duration predictor. It controls speaking rate directly,
#   not via time-stretching, so quality is preserved.
#
# ABOUT pitch_shift:
#   Pitch shifting is done post-synthesis using librosa's
#   phase-vocoder. Each semitone ≈ 5.9% frequency change.
#   Recommended range: -6 to +6 semitones for natural results.

import time, os
import numpy as np
import soundfile as sf
import librosa
from IPython.display import display, HTML, Audio

OUTPUT_PATH = "/content/JARVIS_TTS_output.wav"

# ─── Step 1: Read widget parameters ─────────────────────
voice_name      = w_voice.value
lang            = w_lang.value
total_steps     = w_steps.value
speed           = w_speed.value
pitch_semitones = w_pitch.value
silence_dur     = w_silence.value
chunk_len       = w_chunk.value
verbose         = w_verbose.value

display(HTML(stark_panel(
    log("PARAMS",  f"Voice: <b>{voice_name}</b> | Lang: <b>{lang}</b> | Steps: <b>{total_steps}</b>", "#00BFFF") +
    log("PARAMS",  f"Speed: <b>{speed}x</b> | Pitch: <b>{pitch_semitones:+d} semitones</b>", "#00BFFF") +
    log("PARAMS",  f"Chunk: <b>{chunk_len} chars</b> | Silence: <b>{silence_dur}s</b>", "#00BFFF") +
    log("TEXT",    f"Input length: <b>{len(INPUT_TEXT)} characters</b>", "#00BFFF") +
    log("INIT",    "Loading voice style...", "#FFD700"),
    "REPULSORS CHARGING"
)))

# ─── Step 2: Load voice style ────────────────────────────
if CUSTOM_VOICE_PATH:
    voice_style = tts.get_voice_style_from_path(CUSTOM_VOICE_PATH)
    voice_source = f"Voice Builder JSON: {os.path.basename(CUSTOM_VOICE_PATH)}"
else:
    voice_style = tts.get_voice_style(voice_name=voice_name)
    voice_source = f"Built-in profile: {voice_name}"

print(f"  ⚙  Voice loaded: {voice_source}")

# ─── Step 3: Synthesise ─────────────────────────────────
display(HTML(stark_panel(
    log("VOICE",   voice_source, "#00FF7F") +
    log("FIRING",  "Supertonic-3 ONNX inference starting...", "#CC0000") +
    log("ENGINE",  "text_encoder → vector_estimator → vocoder", "#FFD700") +
    log("NOTE",    "Verbose output will appear in console below the panel", "#00BFFF"),
    "FIRING REPULSORS", "#CC0000"
)))

t_start = time.time()

wav_raw, raw_duration = tts.synthesize(
    text            = INPUT_TEXT,
    voice_style     = voice_style,
    total_steps     = total_steps,
    speed           = speed,
    max_chunk_length= chunk_len,
    silence_duration= silence_dur,
    lang            = lang,
    verbose         = verbose,
)

# ─── FIX: supertonic returns raw_duration as np.ndarray shape (1,) ──────────
# Convert to a plain Python float so f-string formatting works correctly.
raw_duration = float(np.squeeze(raw_duration))

t_synth = time.time() - t_start
rtf = t_synth / raw_duration if raw_duration > 0 else 0.0

print(f"\n  ✅ Raw synthesis complete in {t_synth:.2f}s "
      f"({raw_duration:.2f}s audio, RTF={rtf:.3f}x)")

# ─── Step 4: Pitch shift (post-synthesis) ────────────────
# ─── FIX: wav_raw has shape (1, num_samples); squeeze to 1-D mono array ─────
wav_array = np.squeeze(np.array(wav_raw, dtype=np.float32))
if wav_array.ndim > 1:
    wav_array = np.mean(wav_array, axis=0)  # fallback: average channels → mono

pitch_applied = False
if pitch_semitones != 0:
    print(f"  ⚙  Applying pitch shift: {pitch_semitones:+d} semitones via librosa...")
    t_p = time.time()
    # ─── FIX: use res_type='kaiser_best' (always available; avoids soxr dep) ─
    wav_array = librosa.effects.pitch_shift(
        y=wav_array, sr=SAMPLE_RATE, n_steps=pitch_semitones,
        res_type='kaiser_best',
    )
    print(f"     Pitch shift done in {time.time()-t_p:.2f}s")
    pitch_applied = True

# ─── Step 5: Save output ─────────────────────────────────
sf.write(OUTPUT_PATH, wav_array, SAMPLE_RATE, subtype="PCM_16")
file_size_kb = os.path.getsize(OUTPUT_PATH) / 1024
final_duration = len(wav_array) / SAMPLE_RATE

display(HTML(stark_panel(
    log("SYNTH",   f"Raw synthesis: <b>{raw_duration:.2f}s</b> audio in <b>{t_synth:.2f}s</b> wall-clock", "#00FF7F") +
    log("RTF",     f"Real-Time Factor: <b>{rtf:.4f}x</b>  (lower = faster than real-time)", "#00FF7F") +
    log("SPEED",   f"Speed multiplier applied at synthesis: <b>{speed}x</b>", "#00FF7F") +
    log("PITCH",   (f"Pitch shift applied: <b>{pitch_semitones:+d} semitones</b> via librosa"
                    if pitch_applied else "Pitch shift: <b>none (0 semitones)</b>"), "#00FF7F") +
    log("OUTPUT",  f"File: <b>{OUTPUT_PATH}</b> ({file_size_kb:.1f} KB)", "#00FF7F") +
    log("AUDIO",   f"Final duration: <b>{final_duration:.2f}s</b> @ <b>{SAMPLE_RATE} Hz</b> PCM-16", "#00FF7F") +
    log("STATUS",  "MISSION ACCOMPLISHED — AUDIO READY", "#00FF7F"),
    "SYNTHESIS COMPLETE ─ REPULSORS COOLED", "#00FF7F"
)))

# ─── Step 6: Inline audio player ─────────────────────────
print("\n🔊 Playing generated audio inline:\n")
display(Audio(OUTPUT_PATH, autoplay=False))
print(f"\n✅ Audio saved to {OUTPUT_PATH}. Run Cell 8 to download.\n")


  ⚙  Voice loaded: Built-in profile: M1


📝 Input text length: 527 characters
🌐 Language: hi
Split into 2 chunk(s)
Chunk 1: Yaar, sun toh sahi! <breath> Aaj office mein kuch aisa hua k...
Chunk 2: <sigh> Phir usne bataya ki humara project deadline se poore ...
Synthesizing audio... Settings: steps=8, speed=1.00x, sample_rate=44100Hz
   [1/2] Processing chunk... ✓ (17.84s)
   [2/2] Processing chunk... ✓ (22.79s)
Generation complete!
Total duration: 40.93s
Total samples: 1,810,350
Array shape: (1, 1810350)

  ✅ Raw synthesis complete in 22.62s (40.93s audio, RTF=0.553x)



🔊 Playing generated audio inline:




✅ Audio saved to /content/JARVIS_TTS_output.wav. Run Cell 8 to download.



In [9]:
# ╔══════════════════════════════════════════════════════════╗
# ║   CELL 8 ─ MISSION COMPLETE — DOWNLOAD AUDIO            ║
# ╚══════════════════════════════════════════════════════════╝
# This cell:
#   1. Displays a waveform amplitude preview (ASCII art)
#   2. Shows full synthesis report panel
#   3. Downloads the WAV file to your local machine
#
# The downloaded file is a standard 16-bit PCM WAV at the
# native Supertonic-3 sample rate (24000 Hz by default).
# You can open it in Audacity, VLC, or any DAW.

import numpy as np
import soundfile as sf
import os, time
from IPython.display import display, HTML, Audio
from google.colab import files as colab_files

# ─── Load audio for analysis ────────────────────────────
wav_dl, sr_dl = sf.read(OUTPUT_PATH, dtype="float32")
duration_dl   = len(wav_dl) / sr_dl
peak_db       = 20 * np.log10(np.max(np.abs(wav_dl)) + 1e-9)
rms_db        = 20 * np.log10(np.sqrt(np.mean(wav_dl**2)) + 1e-9)
fsize_kb      = os.path.getsize(OUTPUT_PATH) / 1024

# ─── ASCII waveform preview ─────────────────────────────
N_BINS = 60
HEIGHT = 8
chunk_size = max(1, len(wav_dl) // N_BINS)
bins = [np.max(np.abs(wav_dl[i*chunk_size:(i+1)*chunk_size]))
        for i in range(N_BINS)]
max_bin = max(bins) if max(bins) > 0 else 1
cols = ["#" * int(b / max_bin * HEIGHT) or "." for b in bins]
rows = []
for row in range(HEIGHT, 0, -1):
    line = "".join("█" if len(c) >= row else " " for c in cols)
    rows.append(line)
waveform_ascii = "\n".join(rows)

waveform_html = (
    '<pre style="color:#00FF7F;font-size:.6em;line-height:1.2;'
    'background:#000;padding:8px;border-radius:4px;overflow-x:auto;">'
    + waveform_ascii +
    '\n<span style="color:#555;">└' + '─'*60 + '┘ time →</span>'
    '</pre>'
)

display(HTML(stark_panel(
    log("FILE",     f"<b>{OUTPUT_PATH}</b>", "#00FF7F") +
    log("DURATION", f"<b>{duration_dl:.3f}s</b>", "#00FF7F") +
    log("SAMPLE",   f"<b>{sr_dl} Hz</b>  |  Bit depth: <b>16-bit PCM</b>", "#00FF7F") +
    log("SIZE",     f"<b>{fsize_kb:.1f} KB</b>  ({fsize_kb/1024:.2f} MB)", "#00FF7F") +
    log("PEAK",     f"<b>{peak_db:.1f} dBFS</b>  (0 dBFS = digital full-scale)", "#00FF7F") +
    log("RMS",      f"<b>{rms_db:.1f} dBFS</b>  (average loudness)", "#00FF7F") +
    "<br><b style='color:#00BFFF;font-size:.8em;'>◆ AMPLITUDE WAVEFORM</b>" +
    waveform_html,
    "MISSION DEBRIEF — AUDIO STATS"
)))

# ─── Replay + Download ──────────────────────────────────
display(HTML(stark_panel(
    log("AUDIO",    "Inline player below ─ verify quality before download", "#FFD700") +
    log("DOWNLOAD", "File will save to your browser's default download folder", "#FFD700"),
    "DOWNLOADING PAYLOAD"
)))

display(Audio(OUTPUT_PATH, autoplay=False))

print("\n📥 Initiating download...")
time.sleep(0.5)
colab_files.download(OUTPUT_PATH)

display(HTML(stark_panel(
    log("STATUS",   "Download complete — check your Downloads folder", "#00FF7F") +
    log("FORMAT",   "WAV PCM-16 — compatible with Audacity, VLC, any DAW", "#00BFFF") +
    log("JARVIS",   "Shall I fire another salvo, sir?", "#FFD700"),
    "DOWNLOAD COMPLETE", "#00FF7F"
)))



📥 Initiating download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 🧪 Hinglish Test Paragraph (Copy into your `.txt` file)

Paste the text below into a `.txt` file (UTF-8), upload it in **Cell 4**, and run the notebook to hear it.

> **Expression tags used:** `<laugh>` `<breath>` `<sigh>`
> **Script:** Devanagari (Hindi) + Latin (Hinglish) — Mixed

---

```
Yaar, sun toh sahi! <breath> Aaj office mein kuch aisa hua ki mujhe khud
believe nahi ho raha. मेरा boss अचानक आया और bola, "Tum log bahut achha kaam
kar rahe ho!" <laugh> Main toh sochne laga, yeh koi hidden camera prank hai
kya bhai? <sigh> Phir usne bataya ki humara project deadline se poore 2 din
pehle complete ho gaya, aur management bahut khush hai. Seriously yaar, kabhi
kabhi zindagi mein aisi unexpected surprises aati hain jo दिल खुश कर देती
हैं। <breath> Toh kal evening ko party hai — sab log time pe aana, okay?
```

---

### 📌 Tag Reference

| Tag | Effect | Best used |
|-----|--------|-----------|
| `<laugh>` | Natural laughter sound | After something funny/ironic |
| `<breath>` | Audible breath / mini-pause | Between sentences, before big statements |
| `<sigh>` | Exasperated or content sigh | After disappointment or relief |

### ⚙ Recommended Cell 5 Settings for this text

| Parameter | Recommended Value | Reason |
|-----------|-------------------|--------|
| Language | `na` | Mixed Devanagari + Latin script |
| Voice | `M2` or `F3` | Natural-sounding conversational tone |
| Steps | `8` | Balanced quality vs speed |
| Speed | `1.0` | Normal conversational pace |
| Pitch | `0` | Natural voice unmodified |

### 🌐 Supported Languages (31 total)

`en` `hi` `ko` `ja` `ar` `bg` `cs` `da` `de` `el` `et` `fi` `fr` `hr`
`hu` `id` `it` `lt` `lv` `nl` `pl` `pt` `ro` `ru` `sk` `sl` `es` `sv`
`tr` `uk` `vi` + `na` (unknown/mixed fallback)
